# Initial Data-Quality Inspection

This notebook examines the raw freMTPL2 insurance datasets before any
cleaning or modelling is performed.

The inspection covers:

- dataset dimensions;
- column names and data types;
- missing values;
- duplicate records;
- identifier validity;
- numerical ranges;
- categorical values;
- consistency between policy claim counts and claim-level records.

No raw data is modified in this notebook.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [9]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
DOCS_DIR = PROJECT_ROOT / "docs"

FREQUENCY_PATH = RAW_DATA_DIR / "freMTPL2freq.csv"
SEVERITY_PATH = RAW_DATA_DIR / "freMTPL2sev.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Frequency file exists: {FREQUENCY_PATH.exists()}")
print(f"Severity file exists: {SEVERITY_PATH.exists()}")

Project root: c:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence
Frequency file exists: True
Severity file exists: True


In [10]:
frequency = pd.read_csv(FREQUENCY_PATH)
severity = pd.read_csv(SEVERITY_PATH)

print(f"Frequency shape: {frequency.shape}")
print(f"Severity shape: {severity.shape}")

Frequency shape: (678013, 12)
Severity shape: (26639, 2)


In [11]:
EXPECTED_FREQUENCY_COLUMNS = {
    "IDpol",
    "ClaimNb",
    "Exposure",
    "Area",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "VehBrand",
    "VehGas",
    "Density",
    "Region",
}

EXPECTED_SEVERITY_COLUMNS = {
    "IDpol",
    "ClaimAmount",
}

missing_frequency_columns = (
    EXPECTED_FREQUENCY_COLUMNS - set(frequency.columns)
)

missing_severity_columns = (
    EXPECTED_SEVERITY_COLUMNS - set(severity.columns)
)

unexpected_frequency_columns = (
    set(frequency.columns) - EXPECTED_FREQUENCY_COLUMNS
)

unexpected_severity_columns = (
    set(severity.columns) - EXPECTED_SEVERITY_COLUMNS
)

print("Missing frequency columns:", missing_frequency_columns)
print("Missing severity columns:", missing_severity_columns)
print("Unexpected frequency columns:", unexpected_frequency_columns)
print("Unexpected severity columns:", unexpected_severity_columns)

Missing frequency columns: set()
Missing severity columns: set()
Unexpected frequency columns: set()
Unexpected severity columns: set()


## Dataset Preview

The frequency table should contain one row per policy.

The severity table should contain one row per individual claim. A policy
can therefore appear multiple times in the severity table.

In [12]:
print("Frequency dataset:")
display(frequency.head())

print("Severity dataset:")
display(severity.head())

Frequency dataset:


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1,1,0.100,D,5,0,55,50,B12,'Regular',1217,R82
1,3,1,0.770,D,5,0,55,50,B12,'Regular',1217,R82
2,5,1,0.750,B,6,2,52,50,B12,'Diesel',54,R22
3,10,1,0.090,B,7,0,46,50,B12,'Diesel',76,R72
4,11,1,0.840,B,7,0,46,50,B12,'Diesel',76,R72


Severity dataset:


,IDpol,ClaimAmount
0,1552,995.200
1,1010996,"1,128.120"
2,4024277,"1,851.110"
3,4007252,"1,204.000"
4,4046424,"1,204.000"


In [13]:
frequency_types = pd.DataFrame(
    {
        "column": frequency.columns,
        "data_type": frequency.dtypes.astype(str).values,
    }
)

severity_types = pd.DataFrame(
    {
        "column": severity.columns,
        "data_type": severity.dtypes.astype(str).values,
    }
)

print("Frequency data types:")
display(frequency_types)

print("Severity data types:")
display(severity_types)

Frequency data types:


,column,data_type
0,IDpol,int64
1,ClaimNb,int64
2,Exposure,float64
3,Area,str
4,VehPower,int64
5,VehAge,int64
6,DrivAge,int64
7,BonusMalus,int64
8,VehBrand,str
9,VehGas,str


Severity data types:


,column,data_type
0,IDpol,int64
1,ClaimAmount,float64


In [14]:
def missing_value_summary(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return missing-value counts and percentages."""

    summary = pd.DataFrame(
        {
            "missing_count": dataframe.isna().sum(),
            "missing_percentage": (
                dataframe.isna().mean() * 100
            ),
        }
    )

    return summary.sort_values(
        by="missing_count",
        ascending=False,
    )


frequency_missing = missing_value_summary(frequency)
severity_missing = missing_value_summary(severity)

print("Missing values in frequency data:")
display(frequency_missing)

print("Missing values in severity data:")
display(severity_missing)

Missing values in frequency data:


,missing_count,missing_percentage
IDpol,0,0.000
ClaimNb,0,0.000
Exposure,0,0.000
Area,0,0.000
VehPower,0,0.000
VehAge,0,0.000
DrivAge,0,0.000
BonusMalus,0,0.000
VehBrand,0,0.000
VehGas,0,0.000


Missing values in severity data:


,missing_count,missing_percentage
IDpol,0,0.000
ClaimAmount,0,0.000


## Duplicate Checks

A duplicated policy identifier in the frequency table may be a problem
because that table is expected to contain one row per policy.

Repeated policy identifiers in the severity table are expected because
one policy may have multiple claims.

Completely duplicated rows in the severity table require investigation,
but they should not automatically be deleted. Two claims belonging to
the same policy can legitimately have the same claim amount.

In [15]:
frequency_duplicate_rows = int(frequency.duplicated().sum())
severity_duplicate_rows = int(severity.duplicated().sum())

frequency_duplicate_ids = int(
    frequency["IDpol"].duplicated().sum()
)

frequency_unique_policies = frequency["IDpol"].nunique()
severity_unique_policies = severity["IDpol"].nunique()

print(f"Frequency duplicate rows: {frequency_duplicate_rows:,}")
print(f"Severity duplicate rows: {severity_duplicate_rows:,}")
print(f"Duplicate policy IDs in frequency: {frequency_duplicate_ids:,}")
print(f"Unique frequency policies: {frequency_unique_policies:,}")
print(f"Unique policies with severity records: {severity_unique_policies:,}")

Frequency duplicate rows: 0
Severity duplicate rows: 255
Duplicate policy IDs in frequency: 0
Unique frequency policies: 678,013
Unique policies with severity records: 24,950


In [16]:
frequency_numeric_summary = (
    frequency.select_dtypes(include="number")
    .describe()
    .transpose()
)

severity_numeric_summary = (
    severity.select_dtypes(include="number")
    .describe()
    .transpose()
)

print("Frequency numerical summary:")
display(frequency_numeric_summary)

print("Severity numerical summary:")
display(severity_numeric_summary)

Frequency numerical summary:


,count,mean,std,min,25%,50%,75%,max
IDpol,"678,013.000","2,621,856.921","1,641,782.753",1.000,"1,157,951.000","2,272,152.000","4,046,274.000","6,114,330.000"
ClaimNb,"678,013.000",0.053,0.240,0.000,0.000,0.000,0.000,16.000
Exposure,"678,013.000",0.529,0.364,0.003,0.180,0.490,0.990,2.010
VehPower,"678,013.000",6.455,2.051,4.000,5.000,6.000,7.000,15.000
VehAge,"678,013.000",7.044,5.666,0.000,2.000,6.000,11.000,100.000
DrivAge,"678,013.000",45.499,14.137,18.000,34.000,44.000,55.000,100.000
BonusMalus,"678,013.000",59.762,15.637,50.000,50.000,50.000,64.000,230.000
Density,"678,013.000","1,792.422","3,958.647",1.000,92.000,393.000,"1,658.000","27,000.000"


Severity numerical summary:


,count,mean,std,min,25%,50%,75%,max
IDpol,"26,639.000","2,279,863.831","1,577,201.807",139.000,"1,087,642.500","2,137,413.000","3,180,162.000","6,113,971.000"
ClaimAmount,"26,639.000","2,278.536","29,297.481",1.000,686.810,"1,172.000","1,228.080","4,075,400.560"


In [17]:
variables_to_review = {
    "Exposure": frequency["Exposure"],
    "VehAge": frequency["VehAge"],
    "DrivAge": frequency["DrivAge"],
    "BonusMalus": frequency["BonusMalus"],
    "Density": frequency["Density"],
    "ClaimAmount": severity["ClaimAmount"],
}

percentiles = [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1]

percentile_summary = pd.DataFrame(
    {
        variable: values.quantile(percentiles)
        for variable, values in variables_to_review.items()
    }
)

percentile_summary.index = [
    "Minimum",
    "1st percentile",
    "5th percentile",
    "25th percentile",
    "Median",
    "75th percentile",
    "95th percentile",
    "99th percentile",
    "Maximum",
]

display(percentile_summary)

,Exposure,VehAge,DrivAge,BonusMalus,Density,ClaimAmount
Minimum,0.003,0.000,18.000,50.000,1.000,1.000
1st percentile,0.008,0.000,20.000,50.000,10.000,40.031
5th percentile,0.040,0.000,25.000,50.000,20.000,79.123
25th percentile,0.180,2.000,34.000,50.000,92.000,686.810
Median,0.490,6.000,44.000,50.000,393.000,"1,172.000"
75th percentile,0.990,11.000,55.000,64.000,"1,658.000","1,228.080"
95th percentile,1.000,17.000,72.000,95.000,"7,313.000","4,861.685"
99th percentile,1.000,21.000,80.000,106.000,"27,000.000","16,793.704"
Maximum,2.010,100.000,100.000,230.000,"27,000.000","4,075,400.560"


In [18]:
claim_count_distribution = (
    frequency["ClaimNb"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ClaimNb")
    .reset_index(name="PolicyCount")
)

claim_count_distribution["Percentage"] = (
    claim_count_distribution["PolicyCount"]
    / len(frequency)
    * 100
)

display(claim_count_distribution)

,ClaimNb,PolicyCount,Percentage
0,0,643953,94.976
1,1,32178,4.746
2,2,1784,0.263
3,3,82,0.012
4,4,7,0.001
5,5,2,0.000
6,6,1,0.000
7,8,1,0.000
8,9,1,0.000
9,11,3,0.000


In [19]:
policies_with_claims = int((frequency["ClaimNb"] > 0).sum())
policies_without_claims = int((frequency["ClaimNb"] == 0).sum())

observed_claim_rate = (
    policies_with_claims / len(frequency)
)

print(f"Policies with claims: {policies_with_claims:,}")
print(f"Policies without claims: {policies_without_claims:,}")
print(f"Observed policy claim rate: {observed_claim_rate:.2%}")

Policies with claims: 34,060
Policies without claims: 643,953
Observed policy claim rate: 5.02%


In [20]:
categorical_columns = [
    "Area",
    "VehBrand",
    "VehGas",
    "Region",
]

for column in categorical_columns:
    category_summary = (
        frequency[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
    )

    category_summary["Percentage"] = (
        category_summary["Count"]
        / len(frequency)
        * 100
    )

    print(f"\nDistribution of {column}:")
    display(category_summary)


Distribution of Area:


,Area,Count,Percentage
0,C,191880,28.300
1,D,151596,22.359
2,E,137167,20.231
3,A,103957,15.333
4,B,75459,11.129
5,F,17954,2.648



Distribution of VehBrand:


,VehBrand,Count,Percentage
0,B12,166024,24.487
1,B1,162736,24.002
2,B2,159861,23.578
3,B3,53395,7.875
4,B5,34753,5.126
5,B6,28548,4.211
6,B4,25179,3.714
7,B10,17707,2.612
8,B11,13585,2.004
9,B13,12178,1.796



Distribution of VehGas:


,VehGas,Count,Percentage
0,'Regular',345877,51.013
1,'Diesel',332136,48.987



Distribution of Region:


,Region,Count,Percentage
0,R24,160601,23.687
1,R82,84752,12.500
2,R93,79315,11.698
3,R11,69791,10.293
4,R53,42122,6.213
5,R52,38751,5.715
6,R91,35805,5.281
7,R72,31329,4.621
8,R31,27285,4.024
9,R54,19046,2.809


## Basic Validity Checks

These checks identify clearly impossible or structurally invalid values.

They do not yet define statistical outliers. A value can be unusual
without being invalid.

In [21]:
validity_checks = pd.DataFrame(
    {
        "Check": [
            "Missing frequency policy IDs",
            "Missing severity policy IDs",
            "Non-positive frequency policy IDs",
            "Non-positive severity policy IDs",
            "Negative claim counts",
            "Non-integer claim counts",
            "Non-positive exposure",
            "Negative vehicle ages",
            "Negative driver ages",
            "Negative bonus-malus values",
            "Non-positive density",
            "Non-positive claim amounts",
        ],
        "IssueCount": [
            int(frequency["IDpol"].isna().sum()),
            int(severity["IDpol"].isna().sum()),
            int((frequency["IDpol"] <= 0).sum()),
            int((severity["IDpol"] <= 0).sum()),
            int((frequency["ClaimNb"] < 0).sum()),
            int(
                (
                    ~np.isclose(
                        frequency["ClaimNb"] % 1,
                        0,
                    )
                ).sum()
            ),
            int((frequency["Exposure"] <= 0).sum()),
            int((frequency["VehAge"] < 0).sum()),
            int((frequency["DrivAge"] < 0).sum()),
            int((frequency["BonusMalus"] < 0).sum()),
            int((frequency["Density"] <= 0).sum()),
            int((severity["ClaimAmount"] <= 0).sum()),
        ],
    }
)

display(validity_checks)

,Check,IssueCount
0,Missing frequency policy IDs,0
1,Missing severity policy IDs,0
2,Non-positive frequency policy IDs,0
3,Non-positive severity policy IDs,0
4,Negative claim counts,0
5,Non-integer claim counts,0
6,Non-positive exposure,0
7,Negative vehicle ages,0
8,Negative driver ages,0
9,Negative bonus-malus values,0


In [22]:
frequency_policy_ids = set(frequency["IDpol"])
severity_policy_ids = set(severity["IDpol"])

severity_ids_missing_from_frequency = (
    severity_policy_ids - frequency_policy_ids
)

print(
    "Severity policy IDs missing from frequency data:",
    len(severity_ids_missing_from_frequency),
)

Severity policy IDs missing from frequency data: 6


In [23]:
severity_claim_counts = (
    severity.groupby("IDpol")
    .size()
    .rename("SeverityRowCount")
    .reset_index()
)

claim_count_consistency = frequency[
    ["IDpol", "ClaimNb"]
].merge(
    severity_claim_counts,
    on="IDpol",
    how="outer",
    indicator=True,
)

claim_count_consistency["ClaimNb"] = (
    claim_count_consistency["ClaimNb"]
    .fillna(0)
    .astype(int)
)

claim_count_consistency["SeverityRowCount"] = (
    claim_count_consistency["SeverityRowCount"]
    .fillna(0)
    .astype(int)
)

claim_count_consistency["CountDifference"] = (
    claim_count_consistency["ClaimNb"]
    - claim_count_consistency["SeverityRowCount"]
)

total_claims_from_frequency = int(
    frequency["ClaimNb"].sum()
)

total_claims_from_severity = len(severity)

mismatched_policies = claim_count_consistency[
    claim_count_consistency["CountDifference"] != 0
].copy()

print(
    "Total claims reported by frequency table:",
    f"{total_claims_from_frequency:,}",
)

print(
    "Total claim rows in severity table:",
    f"{total_claims_from_severity:,}",
)

print(
    "Policies with inconsistent claim counts:",
    f"{len(mismatched_policies):,}",
)

Total claims reported by frequency table: 36,102
Total claim rows in severity table: 26,639
Policies with inconsistent claim counts: 9,123


In [24]:
display(mismatched_policies.head(20))

,IDpol,ClaimNb,SeverityRowCount,_merge,CountDifference
0,1,1,0,left_only,1
1,3,1,0,left_only,1
2,5,1,0,left_only,1
3,10,1,0,left_only,1
4,11,1,0,left_only,1
5,13,1,0,left_only,1
6,15,1,0,left_only,1
7,17,1,0,left_only,1
8,18,1,0,left_only,1
9,21,1,0,left_only,1


In [25]:
claim_policies_without_severity = int(
    (
        (claim_count_consistency["ClaimNb"] > 0)
        & (
            claim_count_consistency[
                "SeverityRowCount"
            ] == 0
        )
    ).sum()
)

severity_policies_without_frequency = int(
    (
        claim_count_consistency["_merge"]
        == "right_only"
    ).sum()
)

policies_with_more_severity_rows = int(
    (
        claim_count_consistency[
            "SeverityRowCount"
        ]
        > claim_count_consistency["ClaimNb"]
    ).sum()
)

policies_with_fewer_severity_rows = int(
    (
        claim_count_consistency[
            "SeverityRowCount"
        ]
        < claim_count_consistency["ClaimNb"]
    ).sum()
)

consistency_summary = pd.DataFrame(
    {
        "Measure": [
            "Claims reported in frequency table",
            "Claim rows in severity table",
            "Policies with count mismatches",
            "Claim policies without severity records",
            "Severity policies without frequency records",
            "Policies with more severity rows than ClaimNb",
            "Policies with fewer severity rows than ClaimNb",
        ],
        "Count": [
            total_claims_from_frequency,
            total_claims_from_severity,
            len(mismatched_policies),
            claim_policies_without_severity,
            severity_policies_without_frequency,
            policies_with_more_severity_rows,
            policies_with_fewer_severity_rows,
        ],
    }
)

display(consistency_summary)

,Measure,Count
0,Claims reported in frequency table,36102
1,Claim rows in severity table,26639
2,Policies with count mismatches,9123
3,Claim policies without severity records,9116
4,Severity policies without frequency records,6
5,Policies with more severity rows than ClaimNb,6
6,Policies with fewer severity rows than ClaimNb,9117


In [26]:
quality_summary = pd.DataFrame(
    {
        "Measure": [
            "Frequency rows",
            "Frequency columns",
            "Severity rows",
            "Severity columns",
            "Unique frequency policy IDs",
            "Unique severity policy IDs",
            "Frequency duplicate rows",
            "Severity duplicate rows",
            "Duplicate policy IDs in frequency",
            "Missing values in frequency",
            "Missing values in severity",
            "Policies with claims",
            "Policies without claims",
            "Observed policy claim rate",
            "Claims reported in frequency table",
            "Claim rows in severity table",
            "Policies with claim-count mismatches",
            "Severity policy IDs missing from frequency",
        ],
        "Value": [
            len(frequency),
            frequency.shape[1],
            len(severity),
            severity.shape[1],
            frequency_unique_policies,
            severity_unique_policies,
            frequency_duplicate_rows,
            severity_duplicate_rows,
            frequency_duplicate_ids,
            int(frequency.isna().sum().sum()),
            int(severity.isna().sum().sum()),
            policies_with_claims,
            policies_without_claims,
            observed_claim_rate,
            total_claims_from_frequency,
            total_claims_from_severity,
            len(mismatched_policies),
            len(severity_ids_missing_from_frequency),
        ],
    }
)

display(quality_summary)

,Measure,Value
0,Frequency rows,"678,013.000"
1,Frequency columns,12.000
2,Severity rows,"26,639.000"
3,Severity columns,2.000
4,Unique frequency policy IDs,"678,013.000"
5,Unique severity policy IDs,"24,950.000"
6,Frequency duplicate rows,0.000
7,Severity duplicate rows,255.000
8,Duplicate policy IDs in frequency,0.000
9,Missing values in frequency,0.000


In [27]:
QUALITY_SUMMARY_PATH = (
    REPORTS_DIR / "data_quality_summary.csv"
)

quality_summary.to_csv(
    QUALITY_SUMMARY_PATH,
    index=False,
)

print(f"Saved summary to: {QUALITY_SUMMARY_PATH}")

Saved summary to: c:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\reports\data_quality_summary.csv


In [28]:
findings_document = f"""# Initial Data-Quality Findings

## Dataset Dimensions

- Frequency dataset: {frequency.shape[0]:,} rows and {frequency.shape[1]} columns.
- Severity dataset: {severity.shape[0]:,} rows and {severity.shape[1]} columns.
- Unique policies in frequency data: {frequency_unique_policies:,}.
- Unique policies represented in severity data: {severity_unique_policies:,}.

## Missing Values

- Total missing values in frequency data: {int(frequency.isna().sum().sum()):,}.
- Total missing values in severity data: {int(severity.isna().sum().sum()):,}.

## Duplicate Review

- Completely duplicated frequency rows: {frequency_duplicate_rows:,}.
- Completely duplicated severity rows: {severity_duplicate_rows:,}.
- Duplicated policy identifiers in frequency data: {frequency_duplicate_ids:,}.

Repeated policy identifiers in the severity dataset are expected because
one policy may have multiple claims.

Completely duplicated severity records will not be removed automatically.
Two claims belonging to the same policy may legitimately have the same
claim amount.

## Claim Occurrence

- Policies with at least one reported claim: {policies_with_claims:,}.
- Policies without a reported claim: {policies_without_claims:,}.
- Observed policy claim rate: {observed_claim_rate:.2%}.

The claim-occurrence target is imbalanced because most policies do not
have a claim. Accuracy alone will therefore not be an appropriate model
evaluation metric.

## Cross-Table Consistency

- Total claims according to `ClaimNb`: {total_claims_from_frequency:,}.
- Total rows in the severity dataset: {total_claims_from_severity:,}.
- Policies with inconsistent claim counts: {len(mismatched_policies):,}.
- Severity policy IDs missing from frequency data: {len(severity_ids_missing_from_frequency):,}.
- Claim policies without severity records: {claim_policies_without_severity:,}.

Any inconsistencies will be investigated and documented before the
frequency and severity tables are joined.

## Validity Review

Basic validity checks were performed for:

- missing and non-positive identifiers;
- negative or non-integer claim counts;
- non-positive exposure;
- negative driver or vehicle ages;
- negative bonus-malus values;
- non-positive population density;
- non-positive claim amounts.

Unusual values have not yet been removed. Extreme insurance values may
represent legitimate observations and require further investigation.

## Next Actions

The next data-preparation stage will:

1. determine appropriate data types;
2. resolve or document cross-table inconsistencies;
3. create policy-level claim totals;
4. create derived variables;
5. save cleaned interim datasets;
6. preserve the original raw data unchanged.
"""

FINDINGS_PATH = DOCS_DIR / "data_quality_findings.md"

FINDINGS_PATH.write_text(
    findings_document,
    encoding="utf-8",
)

print(f"Saved findings to: {FINDINGS_PATH}")

Saved findings to: c:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\docs\data_quality_findings.md


In [29]:
if len(mismatched_policies) > 0:
    mismatch_path = (
        REPORTS_DIR
        / "claim_count_mismatches.csv"
    )

    mismatched_policies.to_csv(
        mismatch_path,
        index=False,
    )

    print(f"Saved mismatches to: {mismatch_path}")
else:
    print("No claim-count mismatches were found.")

Saved mismatches to: c:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\reports\claim_count_mismatches.csv
